# PySpark RDD Fundamentals
### SparkContext | RDD | Transformations | Actions | Lazy Evaluation

A beginner-friendly notebook to understand the **core building blocks of Apache Spark** before moving on to DataFrames or Spark SQL.

**What you'll learn:**
1. What Spark is and why RDDs exist
2. SparkContext — the entry point to Spark
3. RDD (Resilient Distributed Dataset)
4. Transformations vs Actions
5. Lazy Evaluation (and why it matters)
6. Hands-on code examples running on 2 CPU cores


## 1. What is Apache Spark?

Apache Spark is a **distributed computing engine** used to process large amounts of data quickly by splitting the work across multiple machines (or CPU cores).

Instead of processing data on a single machine sequentially, Spark:
- Splits data into **partitions**
- Distributes those partitions across **worker nodes / CPU cores**
- Processes them **in parallel**
- Combines the results back together

In this notebook, we simulate a small "cluster" locally using **2 CPU cores** (`local[2]`), which is perfect for learning on Google Colab.


## 2. SparkContext — The Entry Point

**`SparkContext` (commonly `sc`)** is the object that represents the **connection** between your Python program (the *driver*) and the Spark cluster.

### Key points:
- It is the **first thing you create** before doing anything with Spark (using the older RDD API).
- It tells Spark:
  - **Where to run** (`local`, `local[2]`, a cluster URL, etc.)
  - **How many cores/threads** to use
  - What the **application name** is (useful for monitoring)
- Only **one active `SparkContext`** can exist per JVM at a time — that's why we call `sc.stop()` when we're done, especially in notebooks where cells can be re-run.
- Every RDD you create is created **through** the SparkContext (e.g., `sc.parallelize(...)`, `sc.textFile(...)`).

### Analogy:
Think of `SparkContext` like turning on the "engine" of a car. Nothing else — driving, transformations, actions — can happen until the engine (`sc`) is running.

### `local[2]` explained:
| Value | Meaning |
|---|---|
| `local` | Run Spark locally using only 1 thread |
| `local[2]` | Run Spark locally using 2 threads (simulates 2 worker cores) |
| `local[*]` | Run Spark locally using **all** available cores |
| `spark://host:port` | Connect to an actual remote Spark cluster |


## 3. RDD — Resilient Distributed Dataset

An **RDD** is Spark's fundamental data structure — the lowest-level way to represent a distributed collection of data.

### Breaking down the name:
- **Resilient** → Fault-tolerant. If a partition of data is lost (e.g., a worker node crashes), Spark can **recompute it automatically** using lineage information (the record of transformations that built it).
- **Distributed** → The data is **split into partitions** and spread across multiple cores/nodes, allowing parallel processing.
- **Dataset** → It's simply a collection of data — numbers, strings, tuples, rows, etc.

### Key properties of RDDs:
- **Immutable** — once created, an RDD cannot be changed. Any transformation creates a **new** RDD.
- **Lazily evaluated** — transformations don't run immediately (more on this below).
- **Partitioned** — data is automatically split so it can be processed in parallel.


### How to create an RDD:
1. **From an existing collection** in your program:
   ```python
   rdd = sc.parallelize([1, 2, 3, 4, 5])
   ```
2. **From external storage** (a file, HDFS, S3, etc.):
   ```python
   rdd = sc.textFile("data.txt")
   ```


## 4. Transformations vs Actions

Every RDD operation in Spark falls into one of **two categories**:

### 🔄 Transformations
- Operations that produce a **new RDD** from an existing one.
- They are **lazy** — Spark just records *what* to do, it does not execute it yet.
- Examples: `map()`, `filter()`, `flatMap()`, `distinct()`, `union()`, `groupByKey()`, `reduceByKey()`, `sortBy()`

### ⚡ Actions
- Operations that **trigger actual computation** and return a result to the driver program (or write data to storage).
- This is the point where all the "queued up" transformations finally run.
- Examples: `collect()`, `count()`, `first()`, `take(n)`, `reduce()`, `saveAsTextFile()`

### Quick comparison table:

| Aspect | Transformation | Action |
|---|---|---|
| Returns | A new RDD | A value / result (or writes output) |
| Execution | Lazy (not executed immediately) | Eager (executes immediately) |
| Examples | `map`, `filter`, `flatMap` | `collect`, `count`, `reduce`, `take` |
| Triggers computation? | ❌ No | ✅ Yes |

### Simple rule of thumb:
> If the operation's output is **another RDD**, it's a **transformation**.
> If the operation's output is **a value, a list, or writes to disk**, it's an **action**.


## 5. Lazy Evaluation — Why Spark Waits

**Lazy evaluation** means Spark does **not** execute transformations the moment they're written in code. Instead, it builds up a **DAG (Directed Acyclic Graph)** — a step-by-step execution plan — and only runs it when an **action** is called.

### Why does this matter?

1. **Optimization** — Spark can look at the *entire* chain of transformations before running anything, and optimize the execution plan (e.g., combining a `map` and `filter` into a single pass over the data instead of two).
2. **Efficiency** — Spark avoids doing unnecessary work. If you never call an action, the transformations never actually run.
3. **Fault tolerance** — Since Spark just remembers the *lineage* (sequence of transformations), it can recompute lost partitions on demand instead of storing every intermediate result.

### Example of the flow:
```python
rdd2 = rdd1.map(...)       # Lazy - nothing runs
rdd3 = rdd2.filter(...)    # Lazy - nothing runs
result = rdd3.collect()    # ACTION - NOW map() and filter() both execute
```

Until `.collect()` (or another action) is called, Spark has only built a **plan**, not actually touched the data.

### Analogy:
Think of transformations like writing a **recipe** — you're just listing steps. Nothing gets cooked until someone calls "action!" (like a movie director) — that's when the kitchen (Spark) actually starts cooking.


## 6. Summary Cheat Sheet

| Concept | What it is | Key idea |
|---|---|---|
| **SparkContext (`sc`)** | Entry point / connection to Spark | Created once; used to create RDDs |
| **RDD** | Distributed, immutable collection of data | Resilient, Distributed, Dataset |
| **Transformation** | Operation that returns a new RDD | Lazy — builds a plan, doesn't execute |
| **Action** | Operation that returns a result/value | Eager — triggers actual execution |
| **Lazy Evaluation** | Spark delays execution until an action | Enables optimization & efficiency |

---
Now let's see all of this in action with real code, running on **2 CPU cores** in Colab.


In [5]:
# ---------------------------------------------------------
# STEP 1: Install PySpark in Colab
# ---------------------------------------------------------
!pip install pyspark -q   # -q keeps the install output quiet

# ---------------------------------------------------------
# STEP 2: Import required modules
# ---------------------------------------------------------
from pyspark import SparkContext, SparkConf

# ---------------------------------------------------------
# STEP 3: Configure Spark to use 2 CPU cores locally
# "local[2]" tells Spark: run locally using 2 worker threads (2 CPUs)
# ---------------------------------------------------------
conf = SparkConf().setAppName("RDD_Theory_Demo").setMaster("local[2]")

# ---------------------------------------------------------
# STEP 4: Create the SparkContext
# This is the entry point that lets us talk to Spark.
# Nothing else (no RDDs, no transformations, no actions) can
# happen until this is created.
# ---------------------------------------------------------
sc = SparkContext(conf=conf)

# Print confirmation
print("SparkContext created successfully!")
print("Running with:", sc.master)   # Should print local[2]
print("App name:", sc.appName)


SparkContext created successfully!
Running with: local[2]
App name: RDD_Theory_Demo


## 7. Hands-on: RDD, Transformations & Actions

Below, watch closely:
- Cells that create **transformations** (`map`, `filter`) print **nothing** by themselves — they're lazy.
- Only when an **action** (`collect`, `count`, `reduce`, `take`) is called does Spark actually compute something.


In [6]:
# ---------------------------------------------------------
# STEP 1: Create an RDD from a Python list
# parallelize() distributes the data across the 2 CPUs we configured.
# This is how we go from a normal Python list -> a distributed RDD.
# ---------------------------------------------------------
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
rdd = sc.parallelize(numbers)

print("Original RDD (via collect ACTION):", rdd.collect())

# ---------------------------------------------------------
# STEP 2: TRANSFORMATION - map()
# map() applies a function to every element and returns a NEW RDD.
# LAZY: nothing actually executes yet! Spark just remembers the plan.
# ---------------------------------------------------------
squared_rdd = rdd.map(lambda x: x * x)
print("squared_rdd created (but NOT executed yet - it's lazy)")

# ---------------------------------------------------------
# STEP 3: TRANSFORMATION - filter()
# filter() keeps only elements that satisfy a condition.
# Still LAZY - just extends the execution plan (the DAG).
# ---------------------------------------------------------
even_squares_rdd = squared_rdd.filter(lambda x: x % 2 == 0)
print("even_squares_rdd created (still lazy, still not executed)")

# ---------------------------------------------------------
# STEP 4: ACTION - collect()
# THIS is where Spark finally executes map() + filter() together,
# across the 2 CPU cores, and brings the result back to the driver.
# ---------------------------------------------------------
result = even_squares_rdd.glom().collect()
print("\nACTION collect() triggered execution!")
print("Even squares:", result)


Original RDD (via collect ACTION): [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
squared_rdd created (but NOT executed yet - it's lazy)
even_squares_rdd created (still lazy, still not executed)

ACTION collect() triggered execution!
Even squares: [[4, 16], [36, 64, 100]]


In [7]:
# ---------------------------------------------------------
# More ACTIONS - each of these triggers computation immediately
# ---------------------------------------------------------

# count() -> returns the number of elements
total_count = even_squares_rdd.count()
print("Count of even squares:", total_count)

# reduce() -> combines all elements using a function (here: sum)
total_sum = rdd.reduce(lambda a, b: a + b)
print("Sum of original numbers:", total_sum)

# take(n) -> returns just the first n elements (great for peeking at big data)
first_three = rdd.take(3)
print("First 3 elements:", first_three)

# first() -> returns just the very first element
print("First element:", rdd.first())


Count of even squares: 5
Sum of original numbers: 55
First 3 elements: [1, 2, 3]
First element: 1


## 8. Cleaning Up

Always stop the `SparkContext` when you're done. This releases the resources (CPU threads, memory) it was using — good practice especially in notebooks where you might re-run cells.


In [8]:
# ---------------------------------------------------------
# STEP: Stop the SparkContext when done
# Always good practice to release resources.
# ---------------------------------------------------------
sc.stop()
print("SparkContext stopped.")


SparkContext stopped.
